In [ ]:
%pip install flask
%pip install flask_cors
%pip install pandas
%pip install scikit-learn
%pip install imblearn


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from flask_cors import CORS, cross_origin
from flask import Flask, request, jsonify
import pickle

In [2]:
cities = [
    "Mumbai", "Delhi", "Bangalore", "Kolkata", "Chennai", "Hyderabad",
    "Ahmedabad", "Pune", "Surat", "Lucknow", "Jaipur", "Kanpur",
    "Coimbatore", "Indore", "Visakhapatnam", "Bhopal", "Thrissur", "Vadodara"]

# Define device types
device_types = ["low_end_phone", "low_end_laptop", "low_end_tablet","high_end_phone",
                "high_end_laptop",  "high_end_tablet"]

data = pd.read_csv("transaction_data.csv")

data.head()

# Count unique customers
unique_customers = data['customer_id'].nunique()  # Count unique entries in 'customer_id' column

# Filter fraudulent and non-fraudulent transactions (assuming a 'fraudulent' column)
fraudulent_transactions = data[data['fraudulent'] == True].shape[0]  # Count rows where 'fraudulent' is True
non_fraudulent_transactions = data.shape[0] - fraudulent_transactions  # Total rows - fraudulent rows

# Print results
print(f"Total Unique Customers: {unique_customers}")
print(f"Total Fraudulent Transactions: {fraudulent_transactions}")
print(f"Total Non-Fraudulent Transactions: {non_fraudulent_transactions}")

Total Unique Customers: 7404
Total Fraudulent Transactions: 7555
Total Non-Fraudulent Transactions: 32445


In [3]:
# City Encoding (with dictionary for mapping)
city_mapping = {}
for i, city in enumerate(cities):
  city_mapping[i + 1] = city  # Start encoding from 1 (0 can represent 'unknown')
  data["current_city"] = data["current_city"].replace(city, i + 1)
  data["billing_city"] = data["billing_city"].replace(city, i + 1)
  data["shipping_city"] = data["shipping_city"].replace(city, i + 1)

# Device Type Encoding (similar approach)
device_type_mapping = {}
for i, device_type in enumerate(device_types):
  device_type_mapping[i] = device_type
  data["device_type"] = data["device_type"].replace(device_type, i)

# Email and IP Address Encoding (already binary)
data["email"] = data["email"].apply(lambda x: 1 if x.startswith("private_email") else 0)
data["ip_address"] = data["ip_address"].apply(lambda x: 1 if x == "proxy-ip" else 0)

# Customer ID Cleaning
data["customer_id"] = data["customer_id"].str.replace("NEW_CUSTOMER_", "", regex=True)
data["customer_id"] = data["customer_id"].str.replace("CUSTOMER_", "", regex=True)

C:\Users\santh\AppData\Local\Temp\ipykernel_11520\3978286757.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["current_city"] = data["current_city"].replace(city, i + 1)
C:\Users\santh\AppData\Local\Temp\ipykernel_11520\3978286757.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["billing_city"] = data["billing_city"].replace(city, i + 1)
C:\Users\santh\AppData\Local\Temp\ipykernel_11520\3978286757.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain

In [4]:
# # Drop columns from features only (excluding target)
features = data.drop(['fraudulent','transaction_date','billing_city','shipping_city','is_new_device', 'customer_id'], axis=1)
target = 'fraudulent'

# # Extract target variable as a NumPy array
target_array = data['fraudulent'].to_numpy()

# # Split data into training and testing sets (stratify ensures proportional class distribution)
X_train, X_test, y_train, y_test = train_test_split(features, target_array, test_size=0.2, stratify=target_array, random_state=42)
# X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [5]:
# Cost-sensitive learning (modify class_weight if needed)
class_weight = {0: 1, 1: 8}  # Assign higher weight to misclassifying fraudulent transactions (class 1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train Random Forest model
model = RandomForestClassifier(class_weight=class_weight)
model.fit(X_train, y_train)

# Make predictions on the testing set
y_pred = model.predict(X_test)

# Evaluate model performance
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

Precision: 1.0
Recall: 1.0
F1-Score: 1.0


In [6]:
#Dumping pre-trained model, so that we don't have to train everytime a API is called.
pickle.dump(model, open("model_fd.pkl", "wb"))

In [7]:
# 1. Set up Flask app
app = Flask(__name__)
CORS(app, support_credentials=True)

# 2. Load pre-trained model
model = pickle.load(open("model_fd.pkl", "rb"))

# 3. Define API endpoint for prediction
@app.route('/predict', methods=['POST'])
@cross_origin(supports_credentials=True)
def predict():
    try:
        # 4. Receive input data
        json_data = request.json

        # 5. Preprocess input data
        
        data = pd.DataFrame(json_data, index=[0])

        temp_city_mapping = {v: k for k, v in city_mapping.items()}
        temp_device_mapping = {v: k for k, v in device_type_mapping.items()}

        data = data.drop(["customer_id"], axis=1)
        
        data["email"] = data["email"].apply(lambda x: 1 if str(x).startswith("private_email") else 0)
        data["ip_address"] = data["ip_address"].apply(lambda x: 1 if str(x) == "proxy-ip" else 0)
        data["current_city"] = data["current_city"].map(temp_city_mapping)
        data["device_type"] = data["device_type"].map(temp_device_mapping)

        X = data
        X = scaler.transform(X)

        # 6. Make predictions
        predictions = model.predict(X)

        # 7. Return predictions
        return jsonify({'is_fraud': predictions.tolist()[0]},)
    except Exception as e:
        return jsonify({'error': str(e)}), 400

# Run the Flask app
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [04/Apr/2024 17:22:33] "OPTIONS /predict HTTP/1.1" 200 -
127.0.0.1 - - [04/Apr/2024 17:22:33] "POST /predict HTTP/1.1" 200 -
